# Step 3: Bessel Root Enclosures

Locate the positive poloidal (Y) and toroidal (Z) boundary-condition roots, refine pole-free brackets with interval arithmetic, and export the root tables used in Step 4. The paper's basis lemma `lem:lap eig` defines the fields and equations `eq:alphaY` and `eq:alphaZ`; the root lemma `lem:zeros` supplies first-root bounds and uniqueness between poles.

Run top to bottom from `notebooks/` in a fresh Julia kernel. The initial assertion requires `dtype = Interval{Float64}` when [NS_Numerics.jl](../src/NS_Numerics.jl) is loaded.

**Inputs:** the project and routines in [My_Bessel.jl](../src/My_Bessel.jl); no MAT file is read.

**Output:** `data/generated/bessel_zeros.mat`, written without a file-existence guard.

**Consumers:** Parts 4A and 4B of [Step 4](finite_rank_construction.ipynb) load the tables when generating matrices or RHS data. Steps 5–7 use the resulting RHS files.

The search retains roots with dimensionless parameter $\alpha<100$. It visits Y degrees `1:100` and Z degrees `1:99`; the analytic lower bounds exclude higher degrees. Read in order: root equations → degree bounds → complete pole intervals → cutoff classification → packing and export. See the [root certificate](../docs/technical_notes/bessel_zeros_certificate.md) for the proofs.


## Which zeros define the basis?

Let $l\ge1$ be the spherical-harmonic degree, $J_\nu$ the Bessel function of the first kind, and $x>0$ a dimensionless argument. Write $\alpha^Y_{l,j}$ and $\alpha^Z_{l,j}$ for the positive boundary-condition roots in increasing order. These are different from the denominator zeros $\beta_{l,j}$ of $J_{l+1/2}$.

The recurrence forms implemented in `func_Z` and `func_Y` are

$$
f_Z(l,x)=x\frac{J_{l-1/2}(x)}{J_{l+1/2}(x)}-(l+1),
$$

$$
f_Y(l,x)=x\frac{J_{l-1/2}(x)}{J_{l+1/2}(x)}-(l+2)
+\frac{(x^2-l(l-1))(x^2-(l-1)(l+2))}{x^2-l(l-1)(l+2)}\qquad(l\ge2).
$$

For $l=1$, the rational term is replaced by $x^2$. Z roots give toroidal basis elements; Y roots give poloidal elements with coupled Y and X angular channels. Both functions have Bessel poles at $\beta_{l,j}$; Y also has the pole $\gamma_l=\sqrt{l(l-1)(l+2)}$ for $l\ge2$.

A root $\alpha$ gives unit-ball eigenvalue $\alpha^2$. On the physical ball of radius $R_0=32$ used downstream, its eigenvalue is $(\alpha/R_0)^2$. Thus `R_CUT=100` is a cutoff on $\alpha$, not a physical radius or an eigenvalue cutoff of 100. The [root-equation derivation](../docs/technical_notes/bessel_zeros_certificate.md#11-the-two-equations-evaluated-by-the-code) connects these formulas to the paper labels `eq:alphaZ` and `eq:alphaY`.

## Setup, cutoffs, and excluded degrees

`dt` constructs values in the module's numeric type. Here `inf` and `sup` extract interval endpoints, and `rsign` returns a strict positive/negative sign only when the entire value enclosure has that sign; zero means undecidable, not an exact root.

| Parameter | Current value | Role |
|---|---|---|
| `R_CUT` | `100` | Upper cutoff for the dimensionless positive root $\alpha$ |
| `SCAN_STEP` | `3` | Denominator-pole scan width; the assertion requires it to be below $\pi$ |
| `POLE_MARGIN` | `1e-5` | Offset of family-function probes from interval boundaries |
| `L_CAP_Y`, `L_CAP_Z` | `101`, `100` | Exclusive upper limits in the degree loops |

Initial denominator-pole and family-root refinements use the library defaults (`tol=1e-20`, endpoint expansion `δ=1e-10`). The special retry for a family-root enclosure touching or straddling the cutoff uses `δ=0` so it does not re-expand the existing enclosure. The tolerance is only a stopping threshold: an undecidable midpoint may return a wider enclosure. The initial expansion must remain strictly inside the pole-free bracket; changes to the margins or defaults require rechecking that condition.

For degree $l$, `lb` is a lower bound for the first positive family root:

| Family and degree | `lb` | Justification |
|---|---|---|
| Y, $l=1$ | $1$ | The first positive root is at least $2$ |
| Y, $l=2,3$ | $\sqrt{l(l-1)}$ | Strict lower bound from the root lemma’s cubic |
| Y, $l\ge4$ | $l$ | The first positive root is strictly larger than $l$ |
| Z, $l\ge1$ | $\sqrt{l(l+1)}$ | the root lemma’s first-root bound |

For Y, $l\ge100$ gives a first root greater than $100$; for Z, $l\ge100$ gives $l(l+1)>100^2$. The Y loop still visits degree 100 and excludes it immediately via `inf(lb) >= sup(R)`. Degree-range printouts describe loop limits rather than the last nonempty rows. See the [degree-bound derivation](../docs/technical_notes/bessel_zeros_certificate.md#2-which-degrees-and-starting-points-are-needed). Changing `R_CUT` would require revisiting these fixed caps and the boundary checks below.

In [1]:
# Activate the project, load the proof library, and define root-search parameters.
# Run with notebooks/ as the working directory; ../src and ../data are resolved relative to it.
import Pkg
Pkg.activate("..")
include("../src/NS_Numerics.jl")

using .NS_Numerics

# Interval mode is mandatory: the sign / exclusion / enclosure certificates below hold only under it.
@assert dtype == Interval{Float64} "Set dtype = Interval{Float64} in src/NS_Numerics.jl before running this notebook."

const R_CUT = dt(100)            # root cutoff: enclose every Y/Z root in (0, R_CUT)
const SCAN_STEP = dt(3)          # pole-ladder cell width; must be < π (one Bessel zero per cell)
const POLE_MARGIN = dt("1e-5");  # keep sign probes this far off each pole


  Activating 

Using dtype = IntervalArithmetic

project at `~/Changhe/3d-navier-stokes-nonuniqueness-main-run`


.Interval{Float64}
Number of threads = 56


In [2]:
# Define Lemma-5 lower bounds and the degree caps used by the root search.
"""
    y_lower_bound(l) -> dtype

Certified lower bound for the first Y-family root `α^Y_{l,1}` (Lemma 5(3)): returns `l` for `l ≥ 4`
(since `Q_l(l²) < 0 ⟹ α^Y_{l,1} > l`), `√(l(l-1))` for `l = 2, 3`, and `1` for `l = 1`. The value is
provably ≤ the true first root, so it is a safe leftmost probe point — no root lies to its left.
"""
function y_lower_bound(l::Int)
    if l >= 4
        return dt(l)
    elseif l >= 2
        return sqrt(dt(l * (l - 1)))
    else
        return dt(1)
    end
end

"""
    z_lower_bound(l) -> dtype

Certified lower bound `t_l = √(l(l+1))` for the first Z-family root `α^Z_{l,1}` (Lemma 5(3)).
"""
z_lower_bound(l::Int) = sqrt(dt(l * (l + 1)))

# Lemma 5 lower bounds cap the searched degrees: α^Y_{l,1} > l (l ≥ 4) and α^Z_{l,1} ≥ √(l(l+1)).
# Y roots exceed l for l ≥ 4; Z roots are at least sqrt(l(l+1)). Both exclude l ≥ 100.
# The Y loop includes l=100, which search_family_roots excludes before scanning.
const L_CAP_Y = 101
const L_CAP_Z = 100

println("Y-family search degrees: 1:", L_CAP_Y - 1)
println("Z-family search degrees: 1:", L_CAP_Z - 1)

Y-family search degrees: 1:100
Z-family search degrees: 1:99


## Locate poles, then certify every complete root interval

A quotient sign change proves a root only on a continuous pole-free interval.

1. `bessel_pole_ladder` scans from `max(lb,SCAN_STEP)` through the first Bessel pole whose enclosure is strictly above `R_CUT`. Width-3 cells contain at most one pole because consecutive Bessel zeros are more than π apart. Undecidable grid-node signs stop the search.
2. For Y, `add_y_extra_pole!` checks γ against every computed Bessel-pole enclosure, inserts it when `sup(gamma)<inf(terminal_Bessel_pole)`, and the caller keeps the family-pole list through its first pole strictly above the cutoff.
3. `lefts=[lb; sup.(poles[1:end-1])]` and `rights=inf.(poles)` define complete intervals. Probes are moved inward by `POLE_MARGIN`. Every interval must return a strict (+,−) bracket or `search_family_roots` aborts.
4. Local refinement preserves the bracket. Retain only `sup(z)<inf(R)`; retry a cutoff-straddling enclosure with `δ=0`, then discard one wholly at/above R or abort if its position remains undecidable.

Fixed margins may cause a safe failure. They cannot silently skip a required interval. The root lemma gives uniqueness, so successful completion gives the complete below-cutoff list.

See [pole spacing and starting points](../docs/technical_notes/bessel_zeros_certificate.md#bessel-zero-spacing-proof) for the unscanned initial segment.

In [3]:
# J_{l+1/2} in the (l, x) calling convention expected by rigorous_root_interval.
func_J(l, x) = my_besselj(l + 1//2, x)

"""
    bessel_pole_ladder(l; R=R_CUT, x_start=SCAN_STEP) -> Vector{dtype}

Rigorous interval enclosures of the zeros of `J_{l+1/2}` from `x_start` through the first pole
strictly above `R`. Tiles the range into cells of width `SCAN_STEP < π`, so the Bessel-zero spacing theorem
guarantees at most one zero per cell; brackets each cell whose endpoints have certified opposite
signs and refines it with `rigorous_root_interval`. Aborts if any grid node has an undecidable
(interval-straddling) sign, so the returned ladder is complete by construction.
"""
function bessel_pole_ladder(l::Int; R::dtype=R_CUT, x_start::dtype=SCAN_STEP)
    @assert l >= 1
    @assert sup(SCAN_STEP) < inf(Pi)   # cells narrower than π ⟹ at most one Bessel zero each

    n = max(1, ceil(Int, sup((R - x_start) / SCAN_STEP)))
    grid = [x_start + dt(i) * SCAN_STEP for i in 0:n-1]
    append!(grid, R)

    signs = [rsign(func_J(l, x)) for x in grid]
    any(iszero, signs) && error("undecidable J sign for l = $l; shift x_start or reduce SCAN_STEP")

    # Continue past R until the first Bessel pole above it is bracketed.
    while true
        x = grid[end] + SCAN_STEP
        s = rsign(func_J(l, x))
        s == 0 && error("undecidable J sign for l = $l at x = $x")
        push!(grid, x)
        push!(signs, s)
        signs[end-1] * signs[end] < 0 && break
    end

    # Bracket and refine exactly the cells across which J_{l+1/2} changes sign.
    poles = [
        rigorous_root_interval(
            l,
            hull(grid[i], grid[i+1]; dec=:auto),
            func_J
        )
        for i in 1:(length(grid)-1)
        if signs[i] * signs[i+1] < 0
    ]
    inf(poles[end]) > sup(R) || error("cannot certify the first Bessel pole above the cutoff for l = $l")
    return poles
end

"""
    bracket_in_interval(fun, l, a, b) -> (status, pos_pt, neg_pt)

Probe the pole-free interval `(a, b)` for `fun`'s unique Lemma-5 root via the directional sign rule
(`fun > 0` just right of the left pole, `fun → -∞` at the right pole). Evaluates `fun` at
`a + POLE_MARGIN` and `b - POLE_MARGIN` and returns `(:root, pos_pt, neg_pt)` when those signs are
rigorously `+1` then `-1` (a certified bracket), otherwise `(:none, nothing, nothing)`. Aborts on an
undecidable probe sign, so any returned bracket is genuine.
"""
function bracket_in_interval(fun::Function, l::Int, a::dtype, b::dtype)
    left = a + POLE_MARGIN
    right = b - POLE_MARGIN

    sup(left) < inf(right) || error("empty bracket interval for l = $l")

    s_left = rsign(fun(l, left))
    s_right = rsign(fun(l, right))

    s_left == 0 && error("undecidable left sign for l = $l at x = $left")
    s_right == 0 && error("undecidable right sign for l = $l at x = $right")

    if s_left == 1 && s_right == -1
        return :root, left, right
    end

    # Strict signs other than (+,-) do not exclude a root in an omitted probe margin.
    return :none, nothing, nothing
end

"""
    add_y_extra_pole!(poles, l; R=R_CUT) -> poles

Insert the extra Y-family pole `γ = √(l(l-1)(l+2))` (present only for `l ≥ 2`) into the sorted pole
list, if it lies within the search range. Asserts `γ` is disjoint from every Bessel pole — Lemma 5's
hypothesis `γ ≠ β_{l,j}` — aborting on a collision. Mutates and returns `poles`.
"""
function add_y_extra_pole!(poles::Vector{dtype}, l::Int; R::dtype=R_CUT)
    l < 2 && return poles

    gamma = sqrt(dt(l * (l - 1) * (l + 2)))
    for z in poles
        @assert sup(gamma) < inf(z) || sup(z) < inf(gamma) "gamma collides with a Bessel pole for l = $l"
    end

    # R is the terminal Bessel pole when called by search_family_roots.
    if sup(gamma) < inf(R)
        push!(poles, gamma)
        sort!(poles, by=inf)
    end
    return poles
end

"""
    search_family_roots(l, fam; R=R_CUT) -> (roots, status)

All roots of family `fam ∈ (:Y, :Z)` below `R` for degree `l`, returned as rigorous enclosures.
Builds the pole ladder (plus the extra Y pole) through the first family pole strictly above `R`.
Requires a certified root in every full pole interval, using Lemma 5 for uniqueness, then retains
only enclosures strictly below `R`. Cutoff-straddling enclosures are refined or cause an error.
Returns `(dtype[], :excluded_by_lemma5)` when the lower bound excludes the degree; otherwise
returns `(roots, :complete)` only after every full interval has contributed a certified root.
"""
function search_family_roots(l::Int, fam::Symbol;
        R::dtype=R_CUT)

    fun = fam === :Y ? func_Y : fam === :Z ? func_Z : error("unknown family $fam")
    lb = fam === :Y ? y_lower_bound(l) : z_lower_bound(l)

    inf(lb) >= sup(R) && return dtype[], :excluded_by_lemma5

    poles = bessel_pole_ladder(l; R=R, x_start=max(lb, SCAN_STEP))
    fam === :Y && add_y_extra_pole!(poles, l; R=poles[end])
    terminal = findfirst(z -> inf(z) > sup(R), poles)
    poles = poles[1:terminal]

    roots = dtype[]

    # Full pole intervals only; lb safely replaces zero before the first root.
    lefts = [lb; dt.(sup.(poles[1:end-1]))]
    rights = dt.(inf.(poles))

    for (a, b) in zip(lefts, rights)
        st, pos_pt, neg_pt = bracket_in_interval(fun, l, a, b)
        st === :root || error("uncertified root in full pole interval ($a, $b) for $fam, l = $l")

        z = rigorous_root_interval(
            l,
            hull(pos_pt, neg_pt; dec=:auto),
            fun
        )

        if sup(z) >= inf(R) && inf(z) < sup(R)
            # Retry the existing isolator without expanding the certified enclosure.
            z = rigorous_root_interval(l, z, fun; δ=zero(dtype))
        end

        if sup(z) < inf(R)
            push!(roots, z)
        elseif inf(z) < sup(R)
            error("root position relative to the cutoff is undecidable for $fam, l = $l: $z")
        end
        # A certified enclosure with inf(z) >= sup(R) is not exported.
    end

    # Every full interval through the first pole above R has a certified, unique root.
    return roots, :complete
end

search_family_roots

## Interpret a completed search

A successful `search_family_roots` call certifies one unique root in every required complete pole interval through the first pole above the cutoff. It returns `:complete` only after all intervals succeed and each enclosure is classified; otherwise it stops with an error. `:excluded_by_lemma5` records the degree exclusion before scanning. The degree bounds and complete pole scan account for the unsearched portions.

The MAT file stores the root enclosures; the driver retains the statuses and pole search information in memory.

The driver records one list and status per searched degree, then prepends the known Y zero mode at degree1 separately from the positive-root search.

See [refinement and completeness](../docs/technical_notes/bessel_zeros_certificate.md#5-what-does-refinement-establish-and-when-is-the-search-complete).

In [4]:
# Driver: search every degree below the caps; prepend the conventional 0 to the Y row at l = 1.
result_Y = Vector{dtype}[]
status_Y = Symbol[]
result_Z = Vector{dtype}[]
status_Z = Symbol[]

begin
    for l in 1:(L_CAP_Y - 1)
        roots, st = search_family_roots(l, :Y)
        if l == 1
            roots = vcat([dt(0)], roots)  # Conventional zero eigenpair entry used downstream.
        end
        push!(result_Y, roots)
        push!(status_Y, st)
    end

    for l in 1:(L_CAP_Z - 1)
        roots, st = search_family_roots(l, :Z)
        push!(result_Z, roots)
        push!(status_Z, st)
    end
end

# The Y count includes the conventional zero entry; the Z count contains positive roots only.
println("total Y roots: ", sum(length.(result_Y)))
println("total Z roots: ", sum(length.(result_Z)))

total Y roots: 1276
total Z roots: 1241


## Pack the accepted roots and save `bessel_zeros.mat`

`checked_root_table_shape` checks that nonempty rows form a contiguous prefix, later searched rows are empty with an allowed status, and degree 1 has the maximal list length. These checks establish the storage layout; accepting the status `:complete` does not add a root-exclusion argument.

`roots_to_padded_matrix` makes one row per degree $l$, through the last nonempty degree. Positive roots appear in increasing order, with `[0,0]` padding on the right. The Y row at $l=1$ begins with the inserted zero mode from `lem:lap eig`, representing the constant vector field $(\cos\theta,-\sin\theta,0)$ in spherical components. This zero is supplied analytically, without evaluating the singular quotient at zero. There is no Z zero mode.

Consequently the $j$th positive Y root is in column $j+1$ for degree 1 and column $j$ for other degrees; the $j$th positive Z root is always in column $j$. Packed shapes are determined by the last nonempty degree and maximal root-list length after trimming trailing empty rows.

[save_matrix](../src/Matrix_Func.jl) splits the two interval matrices into four Float64 endpoint arrays:

| Field pair | Layout | Meaning |
|---|---|---|
| `Y_zeros_lo`, `Y_zeros_hi` | `(last Y degree, maximal Y list length)` | Lower and upper endpoints of `Y_zeros` |
| `Z_zeros_lo`, `Z_zeros_hi` | `(last Z degree, maximal Z list length)` | Lower and upper endpoints of `Z_zeros` |

The zero mode and padding both have zero endpoints; their positions distinguish them. The file stores neither `status_Y/status_Z` nor the scan parameters or pole lists. The save call does not compare against the existing file or load it back. See the [data layout](../docs/data.md#datageneratedbessel_zerosmat) for the MAT convention; low-level HDF5 readers may display transposed dimensions.

In [5]:
# Pack accepted root enclosures into padded interval matrices for MAT-file storage.
# These assertions check row layout and status labels, not completeness of the root search.
"""
    checked_root_table_shape(results, statuses, fam) -> (L_last, ncol)

Validate that the per-degree root lists can be stored as a padded matrix, and return its shape.
Asserts the nonempty rows form a contiguous prefix `1:L_last`, every higher degree is certified
empty (`:complete` / `:excluded_by_lemma5`), and row 1 holds the maximum number of roots (`ncol`).
"""
function checked_root_table_shape(results, statuses, fam::String)
    L_last = findlast(!isempty, results)
    @assert L_last !== nothing "no $fam roots found"

    for l in 1:L_last
        @assert !isempty(results[l]) "empty row below last nonempty $fam row at l = $l"
    end

    for l in (L_last + 1):length(results)
        @assert isempty(results[l])
        @assert statuses[l] in (:complete, :excluded_by_lemma5)
    end

    ncol = maximum(length.(results))
    @assert ncol == length(results[1]) "$fam row 1 does not contain the maximum number of roots"

    return L_last, ncol
end

"""
    roots_to_padded_matrix(results, statuses, fam) -> Matrix{dtype}

Pack the per-degree root enclosures into an `L_last × ncol` interval matrix (row = degree, ascending
roots), zero-padded on the right with the interval `dt(0)`. Shape is validated up front by
`checked_root_table_shape`.
"""
function roots_to_padded_matrix(results, statuses, fam::String)
    L_last, ncol = checked_root_table_shape(results, statuses, fam)

    roots = fill(dt(0), L_last, ncol)

    for l in 1:L_last
        roots[l, 1:length(results[l])] .= results[l]
    end

    return roots
end

roots_to_padded_matrix

In [6]:
# Save accepted Y/Z root interval matrices to the shared data/generated/bessel_zeros.mat.
# There is no file-existence guard; this call writes the same destination as the original notebook.
Y_zeros = roots_to_padded_matrix(result_Y, status_Y, "Y")
Z_zeros = roots_to_padded_matrix(result_Z, status_Z, "Z")

println("matrix shapes: Y_zeros = ", size(Y_zeros), ", Z_zeros = ", size(Z_zeros))

# save_matrix splits each Interval matrix into Float64 lo/hi arrays: Y_zeros_lo/_hi, Z_zeros_lo/_hi.
save_matrix("../data/generated/bessel_zeros.mat", Dict{String, Any}(
    "Y_zeros" => Y_zeros,
    "Z_zeros" => Z_zeros,
))

matrix shapes: Y_zeros = (96, 32), Z_zeros = (95, 31)


## Load the tables in later steps

`load_besselj_zeros` reconstructs interval matrices from the endpoint pairs, strips trailing padding with `mat_to_lst`, and retains the first-column Y zero. In interval mode, its default locally refines each nonzero root; it assumes the expanded bracket is pole-free and does not prove root-list completeness. `preprocess_para!` then adds the radial coefficients, producing Y entries `[α,A,B]` and Z entries `[α,1]`.

| Consumer | Relationship to root tables |
|---|---|
| [Step 4A](finite_rank_construction.ipynb) | Direct: basis functions for finite-rank matrices and supplied-eigenpair checks |
| [Step 4B](finite_rank_construction.ipynb#step-4b) | Direct: basis reconstruction and derivative bounds for the weighted RHS |
| [Step 5](selfsimilar_linear_operator_finite_rank.ipynb) | Indirect: reads even RHS samples and derivative bounds generated by Step 4B |
| [Step 6](eigen_linear_operator_finite_rank.ipynb) | Indirect: reads odd RHS samples and derivative bounds generated by Step 4B |
| [Step 7: auxiliary field bound](verify_auxiliary_field_bound.ipynb) | Indirect: reads odd RHS samples and bounds for the auxiliary functional source |

The same root table and basis ordering must be used for Step 4 matrices and RHS data.

This notebook stores both root families before the downstream parity selection. The [basis and loading guide](../docs/technical_notes/bessel_basis.md#52-what-loading-and-local-refinement-establish) explains the refinement assumptions and coefficient conventions.